# **Main Notebook**

## Setup and Data Collection

#### Imports and Plot Setup

In [22]:
# Imports
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
import json
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

#### Github Authorization

In [23]:
from config import GITHUB_TOKEN

headers = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

BASE_URL = "https://api.github.com"

#### Check for secrets leak

In [24]:
import re

def check_for_secrets(notebook_path):
    """Check if notebook contains potential secrets"""
    with open(notebook_path, 'r') as f:
        content = f.read()
    
    patterns = [
        r'ghp_[a-zA-Z0-9]{36}',  # GitHub personal access token
        r'github_pat_[a-zA-Z0-9]{22}_[a-zA-Z0-9]{59}',  # GitHub fine-grained token
        r'GITHUB_TOKEN\s*=\s*["\'][^"\']+["\']',  # Hardcoded token assignment
    ]
    
    for pattern in patterns:
        if re.search(pattern, content):
            print("WARNING: Potential secret found!")
            return False
    
    print("No obvious secrets detected")
    return True

check_for_secrets('notebook.ipynb')

No obvious secrets detected


True

#### Function to check Rate limits

In [25]:
def check_rate_limit():
    """Check remaining API calls"""
    response = requests.get(f"{BASE_URL}/rate_limit", headers=headers)
    data = response.json()
    return data['rate']['remaining'], data['rate']['reset']

check_rate_limit()

(3971, 1773416861)

#### Data Collection Functions

In [26]:
def search_repositories(query, sort='stars', order='desc', per_page=100, max_pages=10):
    """
    Search GitHub repositories
    
    Parameters:
    - query: search query (e.g., 'stars:>1000', 'language:python')
    - sort: 'stars', 'forks', 'updated'
    - order: 'asc' or 'desc'
    """
    repos = []
    
    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/search/repositories"
        params = {
            'q': query,
            'sort': sort,
            'order': order,
            'per_page': per_page,
            'page': page
        }
        
        response = requests.get(url, headers=headers, params=params)
        
        if response.status_code == 200:
            data = response.json()
            repos.extend(data['items'])
            
            # Check if there are more pages
            if len(data['items']) < per_page:
                break
                
            # Respect rate limiting
            time.sleep(1)
        else:
            print(f"Error: {response.status_code}")
            break
    
    return repos



#################################################################

def get_repo_languages(owner, repo_name):
    """Get languages used in a repository"""
    url = f"{BASE_URL}/repos/{owner}/{repo_name}/languages"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    return {}



#################################################################

def get_repo_topics(owner, repo_name):
    """Get topics/tags for a repository"""
    url = f"{BASE_URL}/repos/{owner}/{repo_name}/topics"
    headers_topics = headers.copy()
    headers_topics['Accept'] = 'application/vnd.github.mercy-preview+json'
    
    response = requests.get(url, headers=headers_topics)
    
    if response.status_code == 200:
        return response.json().get('names', [])
    return []


################################################################

print("Functions defined successfully!")

Functions defined successfully!


#### Collect Repository Data

In [27]:
print("Collecting top repositories...")

# Collect top starred repos
top_repos = search_repositories(
    query='stars:>5000',  # Adjust threshold as needed
    sort='stars',
    per_page=100,
    max_pages=10  # This gives you up to 1000 repos
)

print(f"Collected {len(top_repos)} repositories")

# Check rate limit
remaining, reset_time = check_rate_limit()
print(f"Rate limit remaining: {remaining}")

KeyboardInterrupt: 

#### Converting to dataframe and extracting info

In [ ]:
df = pd.DataFrame(top_repos)

# Extract key fields
df_clean = pd.DataFrame({
    'name': df['name'],
    'full_name': df['full_name'],
    'owner': df['owner'].apply(lambda x: x['login']),
    'description': df['description'],
    'language': df['language'],
    'stars': df['stargazers_count'],
    'forks': df['forks_count'],
    'watchers': df['watchers_count'],
    'size': df['size'],
    'created_at': pd.to_datetime(df['created_at']),
    'updated_at': pd.to_datetime(df['updated_at']),
    'pushed_at': pd.to_datetime(df['pushed_at']),
    'open_issues': df['open_issues_count'],
    'license': df['license'].apply(lambda x: x['name'] if x else 'No License'),
    'topics': df.get('topics', [[] for _ in range(len(df))]),
    'has_wiki': df['has_wiki'],
    'has_pages': df['has_pages'],
    'archived': df['archived']
})

# Calculate repo age
df_clean['age_days'] = (pd.Timestamp.now(tz='UTC') - df_clean['created_at']).dt.days

# Calculate stars per day (popularity velocity)
df_clean['stars_per_day'] = df_clean['stars'] / (df_clean['age_days'] + 1)

# Save raw data
df_clean.to_csv('github_repos_raw.csv', index=False)
print("Data saved!")

df_clean.head()

Data saved!


,name,full_name,owner,description,language,stars,forks,watchers,size,created_at,updated_at,pushed_at,open_issues,license,topics,has_wiki,has_pages,archived,age_days,stars_per_day
0,build-your-own-x,codecrafters-io/build-your-own-x,codecrafters-io,Master programming by recreating your favorite...,Markdown,474726,44515,474726,1201,2018-05-09 12:03:18+00:00,2026-03-13 15:02:07+00:00,2026-02-21 09:34:54+00:00,442,No License,"[awesome-list, free, programming, tutorial-cod...",False,False,False,2865,165.640614
1,awesome,sindresorhus/awesome,sindresorhus,😎 Awesome lists about all kinds of interesting...,NaN,445182,33522,445182,1534,2014-07-11 13:42:37+00:00,2026-03-13 15:01:03+00:00,2026-03-09 07:59:57+00:00,78,Creative Commons Zero v1.0 Universal,"[awesome, awesome-list, lists, resources, unic...",False,True,False,4263,104.404784
2,freeCodeCamp,freeCodeCamp/freeCodeCamp,freeCodeCamp,freeCodeCamp.org's open-source codebase and cu...,TypeScript,438078,43593,438078,554744,2014-12-24 17:49:19+00:00,2026-03-13 14:59:55+00:00,2026-03-13 14:27:08+00:00,260,"BSD 3-Clause ""New"" or ""Revised"" License","[careers, certification, community, curriculum...",False,False,False,4096,106.926532
3,public-apis,public-apis/public-apis,public-apis,A collective list of free APIs,Python,409004,44187,409004,5012,2016-03-20 23:49:42+00:00,2026-03-13 15:02:46+00:00,2026-02-19 15:14:35+00:00,972,MIT License,"[api, apis, dataset, development, free, list, ...",False,False,False,3644,112.209602
4,free-programming-books,EbookFoundation/free-programming-books,EbookFoundation,:books: Freely available programming books,Python,383945,66007,383945,21056,2013-10-11 06:50:37+00:00,2026-03-13 15:01:39+00:00,2026-03-12 00:48:42+00:00,69,Creative Commons Attribution 4.0 International,"[books, education, hacktoberfest, list, resource]",False,True,False,4536,84.625303


#### Language Details

In [29]:
print("Fetching detailed language information...")

language_details = []
repo_topics = []

for idx, row in df_clean.iterrows():
    if idx % 50 == 0:
        print(f"Processing repo {idx}/{len(df_clean)}")
    
    # Get languages
    langs = get_repo_languages(row['owner'], row['name'])
    language_details.append(langs)
    
    # Get topics
    topics = get_repo_topics(row['owner'], row['name'])
    repo_topics.append(topics)
    
    time.sleep(0.5)  # Rate limiting

df_clean['all_languages'] = language_details
df_clean['topics'] = repo_topics


Fetching detailed language information...
Processing repo 0/1000
Processing repo 50/1000
Processing repo 100/1000
Processing repo 150/1000
Processing repo 200/1000
Processing repo 250/1000
Processing repo 300/1000
Processing repo 350/1000
Processing repo 400/1000
Processing repo 450/1000
Processing repo 500/1000
Processing repo 550/1000
Processing repo 600/1000
Processing repo 650/1000
Processing repo 700/1000
Processing repo 750/1000
Processing repo 800/1000
Processing repo 850/1000
Processing repo 900/1000
Processing repo 950/1000


## Data Processing and Classification

#### Process Language Data

In [30]:
def calculate_language_percentages(lang_dict):
    """Convert language bytes to percentages"""
    if not lang_dict:
        return {}
    total = sum(lang_dict.values())
    return {lang: (bytes/total)*100 for lang, bytes in lang_dict.items()}

df_clean['language_percentages'] = df_clean['all_languages'].apply(calculate_language_percentages)

# Get primary language (highest percentage)
df_clean['primary_language'] = df_clean['language_percentages'].apply(
    lambda x: max(x.keys(), key=lambda k: x[k]) if x else 'Unknown'
)

#### Domain Classification Based on Topics and Description

In [31]:
def classify_repo_domain(row):
    """Classify repository into domains based on topics and description"""
    topics = row['topics'] if isinstance(row['topics'], list) else []
    description = str(row['description']).lower() if row['description'] else ''
    
    # Define domain keywords
    domains = {
        'Machine Learning': ['machine-learning', 'ml', 'deep-learning', 'neural-network', 
                            'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'ai'],
        'Data Science': ['data-science', 'data-analysis', 'pandas', 'numpy', 'jupyter',
                        'analytics', 'visualization', 'matplotlib'],
        'Web Development': ['web', 'frontend', 'backend', 'react', 'vue', 'angular',
                           'django', 'flask', 'express', 'nodejs', 'html', 'css'],
        'Mobile Development': ['android', 'ios', 'mobile', 'react-native', 'flutter',
                              'swift', 'kotlin', 'xamarin'],
        'DevOps': ['devops', 'docker', 'kubernetes', 'ci-cd', 'automation',
                  'infrastructure', 'terraform', 'ansible'],
        'Game Development': ['game', 'unity', 'unreal', 'gamedev', 'gaming'],
        'Cloud': ['aws', 'azure', 'gcp', 'cloud', 'serverless'],
        'Security': ['security', 'cybersecurity', 'encryption', 'authentication'],
        'Database': ['database', 'sql', 'nosql', 'mongodb', 'postgresql', 'mysql'],
        'CLI Tools': ['cli', 'command-line', 'terminal', 'shell'],
        'Framework/Library': ['framework', 'library'],
    }
    
    detected_domains = []
    
    # Check topics
    for domain, keywords in domains.items():
        if any(keyword in topics for keyword in keywords):
            detected_domains.append(domain)
    
    # Check description
    if not detected_domains:
        for domain, keywords in domains.items():
            if any(keyword in description for keyword in keywords):
                detected_domains.append(domain)
    
    return detected_domains if detected_domains else ['Other']

df_clean['domains'] = df_clean.apply(classify_repo_domain, axis=1)

#### Identify Rising Repositories

In [ ]:
# Calculate "rising score" based on recent activity and growth rate

# Repos created in last 2 years
recent_cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=730)
df_clean['is_recent'] = df_clean['created_at'] > recent_cutoff

# Rising score: combination of stars per day and recent update activity
days_since_update = (datetime.now() - df_clean['updated_at']).dt.days
df_clean['activity_score'] = 1 / (days_since_update + 1)  # Higher if recently updated

df_clean['rising_score'] = (
    df_clean['stars_per_day'] * 0.6 + 
    df_clean['activity_score'] * 100 * 0.4
)

# Get top rising repos (created in last 2 years, high rising score)
rising_repos = df_clean[df_clean['is_recent']].nlargest(20, 'rising_score')

TypeError: Invalid comparison between dtype=datetime64[us, UTC] and datetime